----------------- Research Part B  -  Oxford dataset ---------------------------

------------- Data Cleaning -------------------------

In [40]:
# import libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [41]:
yougov = pd.read_csv('cleaned_data_yougov.csv')  # added for part B

Handling missing values

In [42]:
# Drop missing values after transforming spaces into nas.
# Reason is:
# ConfirmedCases and ConfirmedDeaths - every state has cases and deaths, reported from 22 Jan 2020. Data before that date can be removed.
# RegionName - oxcgrt has national level data at the end of the dataset. They should be removed as well. They have blank space in the RegionName col.


# Consider blank spaces in RegionName, ConfirmedCases and ConfirmedDeaths as na values
oxcgrt=pd.read_csv("OxCGRT_AUS_latest.csv", na_values=[" ", "", "NA"], keep_default_na=True)  # " " - A cell that looks empty but actually contains a space character
print(oxcgrt.info())  # summary of data in text form
len(oxcgrt)

<class 'pandas.DataFrame'>
RangeIndex: 9864 entries, 0 to 9863
Data columns (total 61 columns):
 #   Column                                                                           Non-Null Count  Dtype  
---  ------                                                                           --------------  -----  
 0   CountryName                                                                      9864 non-null   str    
 1   CountryCode                                                                      9864 non-null   str    
 2   RegionName                                                                       8768 non-null   str    
 3   RegionCode                                                                       8768 non-null   str    
 4   Jurisdiction                                                                     9864 non-null   str    
 5   Date                                                                             9864 non-null   int64  
 6   C1M_School closing 

9864

In [43]:
# remove national level data from oxcgrt dataset 
oxcgrt = oxcgrt[oxcgrt["Jurisdiction"] != "NAT_TOTAL"]


# convert 'Date' column to datetime format   # turn into functions maybe later
# import datetime 
oxcgrt['Date']=pd.to_datetime(oxcgrt['Date'], format="%Y%m%d")
#oxcgrt['Date']=oxcgrt['Date'].dt.date# remove time  # dt - every row in column 

# print(oxcgrt.head)
# print(oxcgrt.columns)



# remove dates before state level data began
oxcgrt = oxcgrt[oxcgrt["Date"] > "2020-01-21"]


print(f"missing values in ConfirmedCases column:",oxcgrt["ConfirmedCases"].isna().sum())
print(f"missing values in ConfirmedDeaths column:",oxcgrt["ConfirmedDeaths"].isna().sum())
print(f"missing values in RegionName column:",oxcgrt["RegionName"].isna().sum())

# before = len(oxcgrt)

# # dropping nas in RegionName column 
# oxcgrt = oxcgrt.dropna(subset=["RegionName","ConfirmedCases","ConfirmedDeaths"])

# after = len(oxcgrt)

# print("Rows removed:", before - after) 


missing values in ConfirmedCases column: 0
missing values in ConfirmedDeaths column: 0
missing values in RegionName column: 0


In [44]:
print(len(yougov))
print(len(oxcgrt))

39890
8600


There are 8600 data.

In [45]:
# Delete uneccessary columns keeping only 
# 'H6M_Facial Coverings', 'RegionName','Date', 'ConfirmedCases','ConfirmedDeaths'
cols_oxcgrt=['RegionName','Date','H6M_Facial Coverings','ConfirmedCases','ConfirmedDeaths']     # added for part B
oxcgrt=oxcgrt.loc[:,cols_oxcgrt]
oxcgrt

,RegionName,Date,H6M_Facial Coverings,ConfirmedCases,ConfirmedDeaths
21,Australian Capital Territory,2020-01-22,0,0.0,0.0
22,Australian Capital Territory,2020-01-23,0,0.0,0.0
23,Australian Capital Territory,2020-01-24,0,0.0,0.0
24,Australian Capital Territory,2020-01-25,0,0.0,0.0
25,Australian Capital Territory,2020-01-26,0,0.0,0.0
...,...,...,...,...,...
8763,Western Australia,2022-12-27,1,1255595.0,837.0
8764,Western Australia,2022-12-28,1,1255595.0,837.0
8765,Western Australia,2022-12-29,1,1255595.0,837.0
8766,Western Australia,2022-12-30,1,1255595.0,837.0


There are 8600 observations.

In [46]:
oxcgrt.isna().sum()  # added for part B

RegionName              0
Date                    0
H6M_Facial Coverings    0
ConfirmedCases          0
ConfirmedDeaths         0
dtype: int64

0 missing values.

In [47]:
# Complete Rate 

# sorts the rows by: complete_rate = proportion of non-missing values 
# From highest complete data to lowest complete data. So variables with fewer missing values appear first. 

complete_rate = oxcgrt.notna().mean().sort_values(ascending=False)  # added for part B

complete_rate_df = complete_rate.reset_index()
complete_rate_df.columns = ["Variable Name", "Complete Rate"]

complete_rate_df.to_csv("complete_rate_oxcgrt.csv", index=False) # 

In [48]:
complete_rate_df

,Variable Name,Complete Rate
0,RegionName,1.0
1,Date,1.0
2,H6M_Facial Coverings,1.0
3,ConfirmedCases,1.0
4,ConfirmedDeaths,1.0


In [49]:
# checks for duplicate rows
print(oxcgrt.duplicated().any()) 

False


No duplicates are found.

In [ ]:
# create daily cases and deaths as oxcgrt data is cumulative

oxcgrt = oxcgrt.sort_values(["RegionName","Date"])



# groupby("RegionName") is still needed since .diff() needs the place where each state starts and end
oxcgrt["cases_daily"] = oxcgrt.groupby("RegionName")["ConfirmedCases"].diff().fillna(0)# new case per day  (.diff()=current data - previous data)
oxcgrt["deaths_daily"] = oxcgrt.groupby("RegionName")["ConfirmedDeaths"].diff().fillna(0) # new death per day
# There were no missing values except the first data of each state.
# Since only the first date for cases_daily and deaths_daily were missing due to subtraction, nas were filled with 0


# re ordering the columns 
oxcgrt = oxcgrt[["RegionName", "Date", "H6M_Facial Coverings", "ConfirmedCases", "cases_daily", "ConfirmedDeaths", "deaths_daily"]]

oxcgrt.to_csv("oxcgrt_daily_cases_deaths_wNegatives.csv", index=False)
# There should not be any negative values in a cumulative dataset
# there are negative values in the daily cases and deaths from the subtraction occured by .diff() function.
# Reasons can be, errors/data correction - authorities removed duplicate cases or historical records were corrected as the dates are in correct order
# and correct state has been considered
# So, all the negative values in the new dataset, should be checked. (replacing with 0, is wrong).

negative_cases = oxcgrt[oxcgrt["cases_daily"]<0] # negative cases 
negative_cases.to_csv("negative_cases.csv", index=False)

negative_deaths = oxcgrt[oxcgrt["deaths_daily"]<0] # negative deaths 
negative_deaths.to_csv("negative_deaths.csv", index=False)

print(f"Total observations of oxcgrt : ", len(oxcgrt))
print(f"negative cases percentage:", round((oxcgrt["cases_daily"]<0).mean()*100,3))
print(f"negative deaths percentage:", round((oxcgrt["deaths_daily"]<0).mean()*100,3))

Total observations of oxcgrt :  8600
negative cases percentage: 1.36
negative deaths percentage: 0.488


If there was a sorting or grouping error, there should be much larger proportion of negative values. 1.36% of cases
0.49% of deaths, suggests that these can be reporting corrections or errors due to pandemic updates.

In [51]:
# to check whether negative values are spread across all the states 

print(f"neg_cases value counts:", negative_cases["RegionName"].value_counts())
print(f"\nneg_deaths value counts:", negative_deaths["RegionName"].value_counts())

neg_cases value counts: RegionName
Queensland                      27
Northern Territory              23
Western Australia               18
South Australia                 17
Victoria                        14
New South Wales                  8
Australian Capital Territory     7
Tasmania                         3
Name: count, dtype: int64

neg_deaths value counts: RegionName
Western Australia               11
South Australia                  8
Victoria                         7
Northern Territory               6
Australian Capital Territory     4
Queensland                       4
New South Wales                  2
Name: count, dtype: int64


Tasmania doesnt have negative values in daily deaths.

In [52]:
oxcgrt["RegionName"].unique() # no nas

<StringArray>
['Australian Capital Territory',              'New South Wales',
           'Northern Territory',                   'Queensland',
              'South Australia',                     'Tasmania',
                     'Victoria',            'Western Australia']
Length: 8, dtype: str

In [53]:
print(f"neg_cases: ",negative_cases["cases_daily"].describe())
print(f"\nneg_deaths: ",negative_deaths["deaths_daily"].describe())

neg_cases:  count      117.000000
mean     -1011.572650
std       5610.282833
min     -53304.000000
25%        -39.000000
50%         -2.000000
75%         -1.000000
max         -1.000000
Name: cases_daily, dtype: float64

neg_deaths:  count    42.000000
mean     -4.619048
std       8.522431
min     -42.000000
25%      -3.000000
50%      -1.000000
75%      -1.000000
max      -1.000000
Name: deaths_daily, dtype: float64


In neg_cases, min -53304.000000 outlier should be investigated. According to the surrounding data in earlier and later dates, this is not a massive drop in covid severity, but a data correction by authorities or an error.

In [54]:
print(oxcgrt[oxcgrt.isna().any(axis=1)])

Empty DataFrame
Columns: [RegionName, Date, H6M_Facial Coverings, ConfirmedCases, cases_daily, ConfirmedDeaths, deaths_daily]
Index: []


0 missing values.

Create 7 day rolling averages without removing negative values - Method 1

In [55]:
# create 7 day rolling cases and deaths averages before removing negative values

# The rolling average means the average number of new cases reported per day during the preceding week.
# The resulting average values with decimals should not be rounded.

oxcgrt["7days_rolling_cases"] = (oxcgrt.groupby("RegionName")["cases_daily"].rolling(window=7, min_periods=1).mean()
                                 .reset_index(level=0, drop=True))

oxcgrt["7days_rolling_deaths"] = (oxcgrt.groupby("RegionName")["deaths_daily"].rolling(window=7, min_periods=1).mean()
                                  .reset_index(level=0, drop=True))


# re ordering the columns 
oxcgrt = oxcgrt[["RegionName", "Date", "H6M_Facial Coverings", "ConfirmedCases", "cases_daily", "7days_rolling_cases","ConfirmedDeaths", "deaths_daily", "7days_rolling_deaths"]]

oxcgrt.to_csv("oxcgrt__7days_rolling_cases_deaths.csv", index=False)

In [56]:
# check if the negative values have disappeared
oxcgrt.describe()

,Date,H6M_Facial Coverings,ConfirmedCases,cases_daily,7days_rolling_cases,ConfirmedDeaths,deaths_daily,7days_rolling_deaths
count,8600,8600.000000,8.600000e+03,8600.000000,8600.000000,8600.000000,8600.000000,8600.000000
mean,2021-07-12 00:00:00,1.601047,3.186732e+05,1294.384535,1290.947715,503.360581,1.982791,1.975332
min,2020-01-22 00:00:00,0.000000,0.000000e+00,-53304.000000,-965.428571,0.000000,-42.000000,-3.428571
25%,2020-10-16 00:00:00,1.000000,2.350000e+02,0.000000,0.285714,4.000000,0.000000,0.000000
50%,2021-07-12 00:00:00,2.000000,1.792500e+03,1.000000,3.285714,13.000000,0.000000,0.000000
75%,2022-04-07 00:00:00,2.000000,1.384075e+05,308.250000,706.142857,279.250000,0.000000,0.857143
max,2022-12-31 00:00:00,4.000000,3.825104e+06,92264.000000,47304.428571,6405.000000,344.000000,58.714286
std,NaN,1.077753,7.339053e+05,4038.576125,3544.562956,1188.392356,7.391636,4.948294


Minimum value of cases_daily -53304 has been smoothed down to -965.42 after calculating the 7 day rolling average.

In [57]:
negative_7days_rolling_cases = oxcgrt[oxcgrt["7days_rolling_cases"]<0] # negative cases 
negative_7days_rolling_cases.to_csv("negative_7days_rolling_cases.csv", index=False)

negative_7days_rolling_deaths = oxcgrt[oxcgrt["7days_rolling_deaths"]<0] # negative deaths 
negative_7days_rolling_deaths.to_csv("negative_7days_rolling_deaths.csv", index=False)

Create 5 day rolling averages without removing negative values

In [58]:
oxcgrt["5days_rolling_cases"] = (oxcgrt.groupby("RegionName")["cases_daily"].rolling(window=5, min_periods=1)
                           .mean().reset_index(level=0, drop=True))

oxcgrt["5days_rolling_deaths"] = (oxcgrt.groupby("RegionName")["deaths_daily"].rolling(window=5, min_periods=1)
                            .mean().reset_index(level=0, drop=True))


# re ordering the columns 
oxcgrt = oxcgrt[["RegionName", "Date", "H6M_Facial Coverings", "ConfirmedCases", "cases_daily", "7days_rolling_cases", "5days_rolling_cases","ConfirmedDeaths", "deaths_daily", "7days_rolling_deaths","5days_rolling_deaths"]]


oxcgrt.to_csv("oxcgrt__5days_rolling_cases_deaths.csv", index=False)

In [59]:
oxcgrt.describe()

,Date,H6M_Facial Coverings,ConfirmedCases,cases_daily,7days_rolling_cases,5days_rolling_cases,ConfirmedDeaths,deaths_daily,7days_rolling_deaths,5days_rolling_deaths
count,8600,8600.000000,8.600000e+03,8600.000000,8600.000000,8600.000000,8600.000000,8600.000000,8600.000000,8600.000000
mean,2021-07-12 00:00:00,1.601047,3.186732e+05,1294.384535,1290.947715,1291.989953,503.360581,1.982791,1.975332,1.977558
min,2020-01-22 00:00:00,0.000000,0.000000e+00,-53304.000000,-965.428571,-2073.800000,0.000000,-42.000000,-3.428571,-4.800000
25%,2020-10-16 00:00:00,1.000000,2.350000e+02,0.000000,0.285714,0.200000,4.000000,0.000000,0.000000,0.000000
50%,2021-07-12 00:00:00,2.000000,1.792500e+03,1.000000,3.285714,2.600000,13.000000,0.000000,0.000000,0.000000
75%,2022-04-07 00:00:00,2.000000,1.384075e+05,308.250000,706.142857,658.200000,279.250000,0.000000,0.857143,0.600000
max,2022-12-31 00:00:00,4.000000,3.825104e+06,92264.000000,47304.428571,54411.600000,6405.000000,344.000000,58.714286,77.800000
std,NaN,1.077753,7.339053e+05,4038.576125,3544.562956,3592.085425,1188.392356,7.391636,4.948294,5.159301


Minimum value of cases_daily -53304 has been smoothed down to -2073.8 after 5 day rolling average, lesser important value
than the 7 day average value.

Create 6 day rolling averages without removing negative values

In [60]:
# 6 days rolling averages 

oxcgrt["6days_rolling_cases"] = (oxcgrt.groupby("RegionName")["cases_daily"].rolling(window=6, min_periods=1)
                           .mean().reset_index(level=0, drop=True))

oxcgrt["6days_rolling_deaths"] = (oxcgrt.groupby("RegionName")["deaths_daily"].rolling(window=6, min_periods=1)
                            .mean().reset_index(level=0, drop=True))


# re ordering the columns 
oxcgrt = oxcgrt[["RegionName", "Date", "H6M_Facial Coverings", "ConfirmedCases", "cases_daily", "7days_rolling_cases", "6days_rolling_cases","5days_rolling_cases","ConfirmedDeaths", "deaths_daily", "7days_rolling_deaths", "6days_rolling_deaths","5days_rolling_deaths"]]


oxcgrt.to_csv("oxcgrt__6days_rolling_cases_deaths.csv", index=False)

In [61]:
oxcgrt.describe()

,Date,H6M_Facial Coverings,ConfirmedCases,cases_daily,7days_rolling_cases,6days_rolling_cases,5days_rolling_cases,ConfirmedDeaths,deaths_daily,7days_rolling_deaths,6days_rolling_deaths,5days_rolling_deaths
count,8600,8600.000000,8.600000e+03,8600.000000,8600.000000,8600.000000,8600.000000,8600.000000,8600.000000,8600.000000,8600.000000,8600.000000
mean,2021-07-12 00:00:00,1.601047,3.186732e+05,1294.384535,1290.947715,1291.400287,1291.989953,503.360581,1.982791,1.975332,1.976260,1.977558
min,2020-01-22 00:00:00,0.000000,0.000000e+00,-53304.000000,-965.428571,-1143.500000,-2073.800000,0.000000,-42.000000,-3.428571,-4.000000,-4.800000
25%,2020-10-16 00:00:00,1.000000,2.350000e+02,0.000000,0.285714,0.166667,0.200000,4.000000,0.000000,0.000000,0.000000,0.000000
50%,2021-07-12 00:00:00,2.000000,1.792500e+03,1.000000,3.285714,3.000000,2.600000,13.000000,0.000000,0.000000,0.000000,0.000000
75%,2022-04-07 00:00:00,2.000000,1.384075e+05,308.250000,706.142857,689.833333,658.200000,279.250000,0.000000,0.857143,0.833333,0.600000
max,2022-12-31 00:00:00,4.000000,3.825104e+06,92264.000000,47304.428571,50216.833333,54411.600000,6405.000000,344.000000,58.714286,67.166667,77.800000
std,NaN,1.077753,7.339053e+05,4038.576125,3544.562956,3565.160041,3592.085425,1188.392356,7.391636,4.948294,5.034443,5.159301


The 7 day rolling average produced the smallest negative values. Therefore, it is the best solution.

In [62]:
# negative value percentages after 7 day rolling averages 
print(f"7days_rolling_negative cases percentage:", round((oxcgrt["7days_rolling_cases"]<0).mean()*100,3))
print(f"7days_rolling_negative deaths percentage:", round((oxcgrt["7days_rolling_deaths"]<0).mean()*100,3))

7days_rolling_negative cases percentage: 0.337
7days_rolling_negative deaths percentage: 0.36


The percentages are 0.34% cases and 0.36% deaths.

In [63]:
# drop unnecessary columns - 5,6 rolling averages

oxcgrt = oxcgrt.drop(columns=["ConfirmedCases","6days_rolling_cases","5days_rolling_cases","ConfirmedDeaths","6days_rolling_deaths",
    "5days_rolling_deaths"])

In [64]:
oxcgrt.columns

Index(['RegionName', 'Date', 'H6M_Facial Coverings', 'cases_daily',
       '7days_rolling_cases', 'deaths_daily', '7days_rolling_deaths'],
      dtype='str')

In [65]:
# Do not removing negatives ----------------------

# # remove negative values
# oxcgrt["cases_daily"]=oxcgrt["cases_daily"].clip(lower=0)
# oxcgrt["deaths_daily"]=oxcgrt["deaths_daily"].clip(lower=0)
# # probably higher negative values would be clipped not all the negative values, in future. Not sure.



# # oxcgrt.to_csv("oxcgrt_daily_cases_deaths_wNegatives.csv", index=False)
# oxcgrt.to_csv("oxcgrt_new_daily_cases_deaths.csv", index=False)

In [66]:
# print(f"cases: ",oxcgrt["cases_daily"].describe())
# print(f"\ndeaths: ",oxcgrt["deaths_daily"].describe())

In [67]:
oxcgrt.isna().sum() 

RegionName              0
Date                    0
H6M_Facial Coverings    0
cases_daily             0
7days_rolling_cases     0
deaths_daily            0
7days_rolling_deaths    0
dtype: int64

0 missing values

In [68]:
# This is after clipping negative to 0. So, not trying ----------------------------

# # create 7 day rolling cases deaths average

# # The rolling average means the average number of new cases reported per day during the preceding week.
# # The resulting average values with decimals should not be changed. Should not be rounded.

# oxcgrt["rolling_cases"] = (oxcgrt.groupby("RegionName")["cases_daily"].rolling(window=7, min_periods=1)
#                            .mean().reset_index(level=0, drop=True))

# oxcgrt["rolling_deaths"] = (oxcgrt.groupby("RegionName")["deaths_daily"].rolling(window=7, min_periods=1)
#                             .mean().reset_index(level=0, drop=True))


# oxcgrt.to_csv("oxcgrt__rolling_cases_deaths.csv", index=False)

In [69]:
oxcgrt.describe()

,Date,H6M_Facial Coverings,cases_daily,7days_rolling_cases,deaths_daily,7days_rolling_deaths
count,8600,8600.000000,8600.000000,8600.000000,8600.000000,8600.000000
mean,2021-07-12 00:00:00,1.601047,1294.384535,1290.947715,1.982791,1.975332
min,2020-01-22 00:00:00,0.000000,-53304.000000,-965.428571,-42.000000,-3.428571
25%,2020-10-16 00:00:00,1.000000,0.000000,0.285714,0.000000,0.000000
50%,2021-07-12 00:00:00,2.000000,1.000000,3.285714,0.000000,0.000000
75%,2022-04-07 00:00:00,2.000000,308.250000,706.142857,0.000000,0.857143
max,2022-12-31 00:00:00,4.000000,92264.000000,47304.428571,344.000000,58.714286
std,NaN,1.077753,4038.576125,3544.562956,7.391636,4.948294


Taking average over preceding and following days without removing negative values - Method 2

Plus and minus 2 days (5 day centered)

In [70]:
# plus and minus 2 days (a 5 day average)

oxcgrt["5day_centered_cases"] = (oxcgrt.groupby("RegionName")["cases_daily"].rolling(window=5, center=True, min_periods=1)
                           .mean().reset_index(level=0, drop=True))

oxcgrt["5day_centered_deaths"] = (oxcgrt.groupby("RegionName")["deaths_daily"].rolling(window=5, center=True, min_periods=1)
                            .mean().reset_index(level=0, drop=True))


# re ordering the columns 
oxcgrt = oxcgrt[["RegionName", "Date", "H6M_Facial Coverings", "cases_daily", "7days_rolling_cases", "5day_centered_cases","deaths_daily", "7days_rolling_deaths","5day_centered_deaths"]]


oxcgrt.to_csv("oxcgrt__5day_centered_cases_deaths.csv", index=False)

In [71]:
oxcgrt.describe()

,Date,H6M_Facial Coverings,cases_daily,7days_rolling_cases,5day_centered_cases,deaths_daily,7days_rolling_deaths,5day_centered_deaths
count,8600,8600.000000,8600.000000,8600.000000,8600.00000,8600.000000,8600.000000,8600.000000
mean,2021-07-12 00:00:00,1.601047,1294.384535,1290.947715,1295.20100,1.982791,1.975332,1.984225
min,2020-01-22 00:00:00,0.000000,-53304.000000,-965.428571,-2073.80000,-42.000000,-3.428571,-4.800000
25%,2020-10-16 00:00:00,1.000000,0.000000,0.285714,0.20000,0.000000,0.000000,0.000000
50%,2021-07-12 00:00:00,2.000000,1.000000,3.285714,2.60000,0.000000,0.000000,0.000000
75%,2022-04-07 00:00:00,2.000000,308.250000,706.142857,674.60000,0.000000,0.857143,0.600000
max,2022-12-31 00:00:00,4.000000,92264.000000,47304.428571,54411.60000,344.000000,58.714286,77.800000
std,NaN,1.077753,4038.576125,3544.562956,3593.47807,7.391636,4.948294,5.164464


min max values have not changed, it is the same value in rolling average and the centered average. The reason should be explored.

Plus and minus 3 days (7 day centered)

In [72]:
# plus and minus 3 days (a 7 day average)

oxcgrt["7day_centered_cases"] = (oxcgrt.groupby("RegionName")["cases_daily"].rolling(window=7, center=True, min_periods=1)
                           .mean().reset_index(level=0, drop=True))

oxcgrt["7day_centered_deaths"] = (oxcgrt.groupby("RegionName")["deaths_daily"].rolling(window=7, center=True, min_periods=1)
                            .mean().reset_index(level=0, drop=True))


# re ordering the columns 
oxcgrt = oxcgrt[["RegionName", "Date", "H6M_Facial Coverings", "cases_daily", "7days_rolling_cases", "5day_centered_cases", "7day_centered_cases","deaths_daily", "7days_rolling_deaths","5day_centered_deaths","7day_centered_deaths"]]


oxcgrt.to_csv("oxcgrt__7day_centered_cases_deaths.csv", index=False)

In [73]:
oxcgrt.describe()

,Date,H6M_Facial Coverings,cases_daily,7days_rolling_cases,5day_centered_cases,7day_centered_cases,deaths_daily,7days_rolling_deaths,5day_centered_deaths,7day_centered_deaths
count,8600,8600.000000,8600.000000,8600.000000,8600.00000,8600.000000,8600.000000,8600.000000,8600.000000,8600.000000
mean,2021-07-12 00:00:00,1.601047,1294.384535,1290.947715,1295.20100,1294.581285,1.982791,1.975332,1.984225,1.983363
min,2020-01-22 00:00:00,0.000000,-53304.000000,-965.428571,-2073.80000,-965.428571,-42.000000,-3.428571,-4.800000,-3.428571
25%,2020-10-16 00:00:00,1.000000,0.000000,0.285714,0.20000,0.285714,0.000000,0.000000,0.000000,0.000000
50%,2021-07-12 00:00:00,2.000000,1.000000,3.285714,2.60000,3.428571,0.000000,0.000000,0.000000,0.000000
75%,2022-04-07 00:00:00,2.000000,308.250000,706.142857,674.60000,716.285714,0.000000,0.857143,0.600000,0.857143
max,2022-12-31 00:00:00,4.000000,92264.000000,47304.428571,54411.60000,47304.428571,344.000000,58.714286,77.800000,58.714286
std,NaN,1.077753,4038.576125,3544.562956,3593.47807,3545.304241,7.391636,4.948294,5.164464,4.953709


Plus and minus 4 days (9 day centered)

In [74]:
# plus and minus 4 days (a 9 day average)

oxcgrt["9day_centered_cases"] = (oxcgrt.groupby("RegionName")["cases_daily"].rolling(window=9, center=True, min_periods=1)
                           .mean().reset_index(level=0, drop=True))

oxcgrt["9day_centered_deaths"] = (oxcgrt.groupby("RegionName")["deaths_daily"].rolling(window=9, center=True, min_periods=1)
                            .mean().reset_index(level=0, drop=True))


# re ordering the columns 
oxcgrt = oxcgrt[["RegionName", "Date", "H6M_Facial Coverings", "cases_daily", "7days_rolling_cases", "5day_centered_cases", "7day_centered_cases", "9day_centered_cases","deaths_daily", "7days_rolling_deaths","5day_centered_deaths","7day_centered_deaths","9day_centered_deaths"]]


oxcgrt.to_csv("oxcgrt__9day_centered_cases_deaths.csv", index=False)
oxcgrt.describe()

,Date,H6M_Facial Coverings,cases_daily,7days_rolling_cases,5day_centered_cases,7day_centered_cases,9day_centered_cases,deaths_daily,7days_rolling_deaths,5day_centered_deaths,7day_centered_deaths,9day_centered_deaths
count,8600,8600.000000,8600.000000,8600.000000,8600.00000,8600.000000,8600.000000,8600.000000,8600.000000,8600.000000,8600.000000,8600.000000
mean,2021-07-12 00:00:00,1.601047,1294.384535,1290.947715,1295.20100,1294.581285,1294.216369,1.982791,1.975332,1.984225,1.983363,1.982359
min,2020-01-22 00:00:00,0.000000,-53304.000000,-965.428571,-2073.80000,-965.428571,-421.222222,-42.000000,-3.428571,-4.800000,-3.428571,-2.666667
25%,2020-10-16 00:00:00,1.000000,0.000000,0.285714,0.20000,0.285714,0.222222,0.000000,0.000000,0.000000,0.000000,0.000000
50%,2021-07-12 00:00:00,2.000000,1.000000,3.285714,2.60000,3.428571,3.666667,0.000000,0.000000,0.000000,0.000000,0.000000
75%,2022-04-07 00:00:00,2.000000,308.250000,706.142857,674.60000,716.285714,716.361111,0.000000,0.857143,0.600000,0.857143,0.888889
max,2022-12-31 00:00:00,4.000000,92264.000000,47304.428571,54411.60000,47304.428571,44084.222222,344.000000,58.714286,77.800000,58.714286,48.555556
std,NaN,1.077753,4038.576125,3544.562956,3593.47807,3545.304241,3524.020572,7.391636,4.948294,5.164464,4.953709,4.884336


In [75]:
# negative percentages 
print(f"7day_centered_negative cases percentage:", round((oxcgrt["7day_centered_cases"]<0).mean()*100,3))
print(f"7day_centered_negative deaths percentage:", round((oxcgrt["7day_centered_deaths"]<0).mean()*100,3))

print(f"9day_centered_negative cases percentage:", round((oxcgrt["9day_centered_cases"]<0).mean()*100,3))
print(f"9day_centered_negative deaths percentage:", round((oxcgrt["9day_centered_deaths"]<0).mean()*100,3))

7day_centered_negative cases percentage: 0.337
7day_centered_negative deaths percentage: 0.36
9day_centered_negative cases percentage: 0.349
9day_centered_negative deaths percentage: 0.314


In [76]:
oxcgrt.isna().sum()

RegionName              0
Date                    0
H6M_Facial Coverings    0
cases_daily             0
7days_rolling_cases     0
5day_centered_cases     0
7day_centered_cases     0
9day_centered_cases     0
deaths_daily            0
7days_rolling_deaths    0
5day_centered_deaths    0
7day_centered_deaths    0
9day_centered_deaths    0
dtype: int64

Since the results are similar and 9day centered average would lead to data leakage, 7 day rolling avarage can be chosen as the severity measure. 

Since the min max Q1 Q3 and all results are similar, the reason has to be verified.

In [77]:
# Verify the reason for receiving the same values from rolling and centered averages 

# minimum raw value
min_cases_daily =oxcgrt.loc[oxcgrt["cases_daily"].idxmin()]
print(f"min cases_daily:",min_cases_daily)

# minimum 7day rolling value
min_7day_rolling_cases =oxcgrt.loc[oxcgrt["7days_rolling_cases"].idxmin()]
print(f"\nmin 7day rolling average:", min_7day_rolling_cases)

# minimum 7day centered value
min_7day_centered_cases =oxcgrt.loc[oxcgrt["7day_centered_cases"].idxmin()]
print(f"\nmin 7day centered average:", min_7day_centered_cases)

# maximum raw value
max_cases_daily =oxcgrt.loc[oxcgrt["cases_daily"].idxmax()]
print(f"\nmax cases_daily:",max_cases_daily)

# maximum 7day rolling value
max_7day_rolling_cases  =oxcgrt.loc[oxcgrt["7days_rolling_cases"].idxmax()]
print(f"\nmax 7day rolling average:",max_7day_rolling_cases)

# maximum centered value
max_7day_centered_cases  =oxcgrt.loc[oxcgrt["7day_centered_cases"].idxmax()]
print(f"\nmax 7day centered average:",max_7day_centered_cases)

min cases_daily: RegionName                  New South Wales
Date                    2022-01-31 00:00:00
H6M_Facial Coverings                      3
cases_daily                        -53304.0
7days_rolling_cases            13172.285714
5day_centered_cases                  1469.6
7day_centered_cases             5377.571429
9day_centered_cases            10672.666667
deaths_daily                           29.0
7days_rolling_deaths              40.428571
5day_centered_deaths                   34.4
7day_centered_deaths                   41.0
9day_centered_deaths              37.777778
Name: 1857, dtype: object

min 7day rolling average: RegionName                  South Australia
Date                    2022-02-15 00:00:00
H6M_Facial Coverings                      2
cases_daily                           103.0
7days_rolling_cases             -965.428571
5day_centered_cases                  1103.8
7day_centered_cases             1415.857143
9day_centered_cases                  1278.0
deaths

NSW and SA have all the min and max values. NSW has all the max values. 

Check the correlation 

In [78]:
oxcgrt[["cases_daily","7days_rolling_cases","7day_centered_cases"]].corr()

,cases_daily,7days_rolling_cases,7day_centered_cases
cases_daily,1.000000,0.870183,0.885899
7days_rolling_cases,0.870183,1.000000,0.974133
7day_centered_cases,0.885899,0.974133,1.000000


Although the values are not identical, they move together very closely. Thats why the results are similar.
Smoothing changes the raw data it can be seen from the different correlations with cases_daily but the trend is very similar.

Check where do they differ

In [79]:
# oxcgrt = pd.read_csv('oxcgrt__9day_centered_cases_deaths.csv')  # added for part B

difference= (oxcgrt["7day_centered_cases"] - oxcgrt["7days_rolling_cases"])

oxcgrt["abs_difference"] = difference.abs()

oxcgrt.nlargest(n=50, columns="abs_difference")

oxcgrt.to_csv("rolling_centered_difference.csv", index=False)


In [80]:
oxcgrt.nlargest(n=50, columns="abs_difference")

,RegionName,Date,H6M_Facial Coverings,cases_daily,7days_rolling_cases,5day_centered_cases,7day_centered_cases,9day_centered_cases,deaths_daily,7days_rolling_deaths,5day_centered_deaths,7day_centered_deaths,9day_centered_deaths,abs_difference
1844,New South Wales,2022-01-18,3,32297.0,46970.285714,29285.8,28674.857143,29926.777778,32.0,25.142857,31.2,29.428571,30.000000,18295.428571
7312,Victoria,2022-01-06,2,21474.0,13968.000000,31162.8,29311.714286,28071.000000,6.0,5.714286,7.2,5.714286,6.222222,15343.714286
1843,New South Wales,2022-01-17,3,29830.0,47304.428571,31222.6,32731.142857,34591.777778,36.0,23.571429,26.0,28.000000,28.333333,14573.285714
7311,Victoria,2022-01-05,2,21790.0,11740.571429,25180.6,25459.142857,24531.333333,6.0,6.000000,6.8,5.857143,5.111111,13718.571429
7313,Victoria,2022-01-07,2,51089.0,20214.714286,34726.4,32891.285714,31772.666667,9.0,5.714286,5.4,7.285714,8.222222,12676.571429
1856,New South Wales,2022-01-30,3,18306.0,23460.428571,2933.6,11029.000000,11752.222222,27.0,40.428571,43.6,41.571429,38.888889,12431.428571
1837,New South Wales,2022-01-11,2,34636.0,32687.285714,47045.8,44756.857143,43655.222222,21.0,13.000000,20.2,19.571429,18.444444,12069.571429
1836,New South Wales,2022-01-10,2,25658.0,32747.000000,40539.2,44315.428571,44084.222222,11.0,11.285714,17.6,18.000000,17.444444,11568.428571
1845,New South Wales,2022-01-19,3,30187.0,38102.142857,27385.0,26664.571429,26297.000000,25.0,25.571429,33.8,32.857143,29.333333,11437.571429
1855,New South Wales,2022-01-29,2,18262.0,23054.571429,10623.8,11827.857143,12921.111111,51.0,38.571429,45.2,40.142857,38.666667,11226.714286


In [81]:
oxcgrt["abs_difference"].describe()

count     8600.000000
mean       178.775777
std        786.238587
min          0.000000
25%          0.000000
50%          0.714286
75%         54.428571
max      18295.428571
Name: abs_difference, dtype: float64